# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [1]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions

import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions

import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [2]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [3]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset2"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset2"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset2
✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset2


In [4]:
!ls -a

.		    .streamlit
..		    .vscode
.bash_history	    app.py
.bashrc		    ce712-yolov8-Copy1.ipynb
.cache		    ce712-yolov8.ipynb
.config		    damage_distribution.png
.docker		    heatmap_1_hurricane-michael_00000304_post_disaster.png
.gemini		    heatmap_2_woolsey-fire_00000348_post_disaster.png
.gsutil		    heatmap_3_socal-fire_00001054_post_disaster.png
.ipynb_checkpoints  runs
.ipython	    vista-ai
.jupyter	    xbd-dataset
.jupyter_ystore.db  xbd-dataset.zip
.lastactive	    xview2-challenge-dataset-train-and-test
.local		    xview2-challenge-dataset-train-and-test.zip
.npm		    yolo26n.pt
.nv		    yolov8n.pt
.ssh


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [5]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [6]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined
✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [7]:
DATASET_BASE = "xbd-dataset/xbd" # Substitui pelo teu caminho real
OUTPUT_DIR = "visual-aid/yolo_dataset2" # Substitui pelo teu caminho real

In [8]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def augment_image_color(image):
    """
    Applies a random selection of color-based augmentations that do not affect
    bounding box coordinates.
    Args:
        image: An image in BGR format (as read by cv2.imread).
    Returns:
        The augmented image in BGR format.
    """
    augmented_image = image.copy()

    # --- 1. Brightness & Contrast ---
    if random.random() > 0.5:
        alpha = random.uniform(0.75, 1.25)  # Contrast factor
        beta = random.randint(-25, 25)     # Brightness shift
        augmented_image = cv2.convertScaleAbs(augmented_image, alpha=alpha, beta=beta)

    # --- 2. Saturation (HSV color space) ---
    if random.random() > 0.5:
        hsv = cv2.cvtColor(augmented_image, cv2.COLOR_BGR2HSV)
        h, s, v = cv2.split(hsv)
        saturation_scale = random.uniform(0.8, 1.2)
        s = np.clip(s * saturation_scale, 0, 255).astype(np.uint8)
        hsv = cv2.merge([h, s, v])
        augmented_image = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    # --- 3. Gaussian Noise ---
    if random.random() > 0.5:
        row, col, ch = augmented_image.shape
        mean = 0
        var = random.uniform(10, 50)
        sigma = var**0.5
        gauss = np.random.normal(mean, sigma, (row, col, ch))
        noisy = np.clip(augmented_image + gauss, 0, 255)
        augmented_image = noisy.astype(np.uint8)

    return augmented_image

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file(s)
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output directories
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # --- OVERSAMPLING & AUGMENTATION LOGIC ---
                num_copies = 1
                # Only oversample the training set to avoid biased validation metrics
                if split_name == 'train':
                    # Check for minority classes
                    contains_moderate = any(ann['yolo_class'] == 1 for ann in annotations)
                    contains_destruction = any(ann['yolo_class'] == 2 for ann in annotations)

                    # Define oversampling factors
                    oversample_moderate = 3
                    oversample_destruction = 5

                    if contains_destruction:
                        num_copies = oversample_destruction
                    elif contains_moderate:
                        num_copies = oversample_moderate

                # Create original and copies
                for i in range(num_copies):
                    # For copies (i > 0), create a new unique filename
                    if i > 0:
                        base, ext = os.path.splitext(img_file)
                        current_img_file = f"{base}_aug{i}{ext}"
                        output_img_path = os.path.join(output_img_dir, current_img_file)

                        original_image = cv2.imread(img_path)
                        if original_image is not None:
                            augmented_image = augment_image_color(original_image)
                            cv2.imwrite(output_img_path, augmented_image)
                        else: # Fallback if image read fails
                            shutil.copy2(img_path, output_img_path)
                    else: # For the original image (i=0), just copy it
                        current_img_file = img_file

                    output_img_path = os.path.join(output_img_dir, current_img_file)
                    shutil.copy2(img_path, output_img_path)

                    # Create corresponding label file
                    label_file = current_img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                    label_path = os.path.join(output_label_dir, label_file)

                    with open(label_path, 'w') as f:
                        for ann in annotations:
                            yolo_class = ann['yolo_class']
                            bbox = ann['bbox']
                            yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                            if yolo_bbox is not None:
                                f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\\n")
                                # Update stats for every instance written
                                stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [2:19:52<00:00,  1.24s/it]  



Processing val split...


Converting val: 100%|██████████| 1699/1699 [08:44<00:00,  3.24it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3986
  Class distribution:
    No-Damage (class 0): 421969
    Moderate-Damage (class 1): 141069
    Total-Destruction (class 2): 62115

VAL:
  Total images: 1699
  Images with annotations: 960
  Class distribution:
    No-Damage (class 0): 34700
    Moderate-Damage (class 1): 7395
    Total-Destruction (class 2): 5776

✅ Dataset conversion completed!


In [14]:
import os
import glob
from multiprocessing import Pool, cpu_count

def fix_label_file(filepath):
    """Lê o arquivo, corrige as quebras de linha literais e salva."""
    try:
        with open(filepath, 'r') as f:
            content = f.read()
        
        # Verifica se existe a string literal "\n"
        if r"\n" in content:
            fixed_content = content.replace(r"\n", "\n")
            
            with open(filepath, 'w') as f:
                f.write(fixed_content)
            return True # Retorna True se o arquivo foi modificado
        return False
    except Exception as e:
        print(f"Erro ao processar {filepath}: {e}")
        return False

# Em notebooks e scripts Python, é boa prática encapsular a execução principal
# para evitar que os processos filhos tentem reexecutar o código todo recursivamente.
if __name__ == '__main__':
    # Ajuste para o caminho da sua pasta de labels (com o asterisco no final)
    labels_dir = "visual-aid/yolo_dataset2/labels/train/*.txt" 
    filepaths = glob.glob(labels_dir)
    
    # Descobre quantos núcleos de CPU estão disponíveis no ambiente
    num_cores = cpu_count()
    print(f"Iniciando correção em paralelo usando {num_cores} núcleos para {len(filepaths)} arquivos...")
    
    # Cria o pool de processos paralelos
    with Pool(processes=num_cores) as pool:
        # pool.map aplica a função 'fix_label_file' a cada item da lista 'filepaths'
        results = pool.map(fix_label_file, filepaths)
        
    # Conta quantos arquivos retornaram 'True' (foram de fato alterados)
    arquivos_corrigidos = sum(1 for r in results if r)
    
    print(f"✅ Processamento concluído! {arquivos_corrigidos} arquivos foram corrigidos com sucesso.")

Iniciando correção em paralelo usando 8 núcleos para 11810 arquivos...
✅ Processamento concluído! 9941 arquivos foram corrigidos com sucesso.


In [15]:
def fix_label_file(filepath):
    """Lê o arquivo, corrige as quebras de linha literais e salva."""
    try:
        with open(filepath, 'r') as f:
            content = f.read()
        
        # Verifica se existe a string literal "\n"
        if r"\n" in content:
            fixed_content = content.replace(r"\n", "\n")
            
            with open(filepath, 'w') as f:
                f.write(fixed_content)
            return True # Retorna True se o arquivo foi modificado
        return False
    except Exception as e:
        print(f"Erro ao processar {filepath}: {e}")
        return False

# Em notebooks e scripts Python, é boa prática encapsular a execução principal
# para evitar que os processos filhos tentem reexecutar o código todo recursivamente.
if __name__ == '__main__':
    # Ajuste para o caminho da sua pasta de labels (com o asterisco no final)
    labels_dir = "visual-aid/yolo_dataset2/labels/val/*.txt" 
    filepaths = glob.glob(labels_dir)
    
    # Descobre quantos núcleos de CPU estão disponíveis no ambiente
    num_cores = cpu_count()
    print(f"Iniciando correção em paralelo usando {num_cores} núcleos para {len(filepaths)} arquivos...")
    
    # Cria o pool de processos paralelos
    with Pool(processes=num_cores) as pool:
        # pool.map aplica a função 'fix_label_file' a cada item da lista 'filepaths'
        results = pool.map(fix_label_file, filepaths)
        
    # Conta quantos arquivos retornaram 'True' (foram de fato alterados)
    arquivos_corrigidos = sum(1 for r in results if r)
    
    print(f"✅ Processamento concluído! {arquivos_corrigidos} arquivos foram corrigidos com sucesso.")

Iniciando correção em paralelo usando 8 núcleos para 960 arquivos...
✅ Processamento concluído! 960 arquivos foram corrigidos com sucesso.


In [11]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


✅ Created data.yaml at: visual-aid/yolo_dataset2/data.yaml

Configuration:
# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: /home/jupyter/visual-aid/yolo_dataset2  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes



## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 1
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


🚀 TRAINING PREPARATION

📊 Environment Check (CRITICAL for debugging):
   NumPy: 1.26.4
   OpenCV: 4.8.1
   ✅ Versions compatible!

✅ Dataset configuration found: visual-aid/yolo_dataset2/data.yaml

📁 Dataset Structure Check:
   Train images: visual-aid/yolo_dataset2/images/train - ✅
   Val images: visual-aid/yolo_dataset2/images/val - ✅
   Train labels: visual-aid/yolo_dataset2/labels/train - ✅
   Val labels: visual-aid/yolo_dataset2/labels/val - ✅
   Train images count: 11810
   Val images count: 960

🤖 Initializing YOLOv26n model...
   ✅ Model loaded successfully

🚀 Starting training...
   Model: YOLOv26n
   Epochs: 1
   Batch Size: 16
   Image Size: 640

⚠️  DEBUGGING INFO:
   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible
   Current: NumPy 1.26.4, OpenCV 4.8.1
   Solution: Restart kernel + run Cell 1 to fix versions

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22478MiB)
engine/trainer: agnostic_nms=False, amp=False, 

## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)


# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [22]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions
import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [23]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [ ]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset2"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset2"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset


In [25]:
!ls -a

.
..
.bashrc
.cache
.config
.docker
.gemini
.ipynb_checkpoints
.ipython
.jupyter
.jupyter_ystore.db
.jupyter_ystore.db-journal
.lastactive
.local
.npm
.nv
Untitled.ipynb
ce712-yolov8.ipynb
heatmap_1_hurricane-michael_00000304_post_disaster.png
heatmap_2_socal-fire_00001054_post_disaster.png
heatmap_3_hurricane-michael_00000500_post_disaster.png
notebook_template.ipynb
runs
visual-aid
xbd-dataset
xbd-dataset.zip
xview2-challenge-dataset-train-and-test
xview2-challenge-dataset-train-and-test.zip
yolo_dataset
yolov8n.pt


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [26]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [ ]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [ ]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [09:48<00:00, 11.54it/s]



Processing val split...


Converting val: 100%|██████████| 1699/1699 [02:30<00:00, 11.30it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3962
  Class distribution:
    No-Damage (class 0): 162251
    Moderate-Damage (class 1): 33263
    Total-Destruction (class 2): 13611

VAL:
  Total images: 1699
  Images with annotations: 984
  Class distribution:
    No-Damage (class 0): 41008
    Moderate-Damage (class 1): 9287
    Total-Destruction (class 2): 4588

✅ Dataset conversion completed!


In [30]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


✅ Created data.yaml at: ./yolo_dataset/data.yaml

Configuration:
# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: /home/jupyter/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes



In [ ]:
print(OUTPUT_DIR)

## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)


# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [22]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions
import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [23]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [ ]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset2"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset2"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset


In [25]:
!ls -a

.
..
.bashrc
.cache
.config
.docker
.gemini
.ipynb_checkpoints
.ipython
.jupyter
.jupyter_ystore.db
.jupyter_ystore.db-journal
.lastactive
.local
.npm
.nv
Untitled.ipynb
ce712-yolov8.ipynb
heatmap_1_hurricane-michael_00000304_post_disaster.png
heatmap_2_socal-fire_00001054_post_disaster.png
heatmap_3_hurricane-michael_00000500_post_disaster.png
notebook_template.ipynb
runs
visual-aid
xbd-dataset
xbd-dataset.zip
xview2-challenge-dataset-train-and-test
xview2-challenge-dataset-train-and-test.zip
yolo_dataset
yolov8n.pt


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [26]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [ ]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [ ]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [09:48<00:00, 11.54it/s]



Processing val split...


Converting val: 100%|██████████| 1699/1699 [02:30<00:00, 11.30it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3962
  Class distribution:
    No-Damage (class 0): 162251
    Moderate-Damage (class 1): 33263
    Total-Destruction (class 2): 13611

VAL:
  Total images: 1699
  Images with annotations: 984
  Class distribution:
    No-Damage (class 0): 41008
    Moderate-Damage (class 1): 9287
    Total-Destruction (class 2): 4588

✅ Dataset conversion completed!


In [30]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


✅ Created data.yaml at: ./yolo_dataset/data.yaml

Configuration:
# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: /home/jupyter/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes



## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


🚀 TRAINING PREPARATION

📊 Environment Check (CRITICAL for debugging):
   NumPy: 1.26.4
   OpenCV: 4.8.1
   ✅ Versions compatible!

✅ Dataset configuration found: ./yolo_dataset/data.yaml

📁 Dataset Structure Check:
   Train images: ./yolo_dataset/images/train - ✅
   Val images: ./yolo_dataset/images/val - ✅
   Train labels: ./yolo_dataset/labels/train - ✅
   Val labels: ./yolo_dataset/labels/val - ✅
   Train images count: 4337
   Val images count: 1325

🤖 Initializing YOLOv26n model...
   ✅ Model loaded successfully

🚀 Starting training...
   Model: YOLOv26n
   Epochs: 10000
   Batch Size: 16
   Image Size: 640

⚠️  DEBUGGING INFO:
   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible
   Current: NumPy 1.26.4, OpenCV 4.8.1
   Solution: Restart kernel + run Cell 1 to fix versions

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22478MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaug

## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)


# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [22]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions
import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [23]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [ ]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset2"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset2"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset


In [25]:
!ls -a

.
..
.bashrc
.cache
.config
.docker
.gemini
.ipynb_checkpoints
.ipython
.jupyter
.jupyter_ystore.db
.jupyter_ystore.db-journal
.lastactive
.local
.npm
.nv
Untitled.ipynb
ce712-yolov8.ipynb
heatmap_1_hurricane-michael_00000304_post_disaster.png
heatmap_2_socal-fire_00001054_post_disaster.png
heatmap_3_hurricane-michael_00000500_post_disaster.png
notebook_template.ipynb
runs
visual-aid
xbd-dataset
xbd-dataset.zip
xview2-challenge-dataset-train-and-test
xview2-challenge-dataset-train-and-test.zip
yolo_dataset
yolov8n.pt


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [26]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [ ]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [ ]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [09:48<00:00, 11.54it/s]



Processing val split...


Converting val: 100%|██████████| 1699/1699 [02:30<00:00, 11.30it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3962
  Class distribution:
    No-Damage (class 0): 162251
    Moderate-Damage (class 1): 33263
    Total-Destruction (class 2): 13611

VAL:
  Total images: 1699
  Images with annotations: 984
  Class distribution:
    No-Damage (class 0): 41008
    Moderate-Damage (class 1): 9287
    Total-Destruction (class 2): 4588

✅ Dataset conversion completed!


In [30]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


✅ Created data.yaml at: ./yolo_dataset/data.yaml

Configuration:
# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: /home/jupyter/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes



## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


🚀 TRAINING PREPARATION

📊 Environment Check (CRITICAL for debugging):
   NumPy: 1.26.4
   OpenCV: 4.8.1
   ✅ Versions compatible!

✅ Dataset configuration found: ./yolo_dataset/data.yaml

📁 Dataset Structure Check:
   Train images: ./yolo_dataset/images/train - ✅
   Val images: ./yolo_dataset/images/val - ✅
   Train labels: ./yolo_dataset/labels/train - ✅
   Val labels: ./yolo_dataset/labels/val - ✅
   Train images count: 4337
   Val images count: 1325

🤖 Initializing YOLOv26n model...
   ✅ Model loaded successfully

🚀 Starting training...
   Model: YOLOv26n
   Epochs: 10000
   Batch Size: 16
   Image Size: 640

⚠️  DEBUGGING INFO:
   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible
   Current: NumPy 1.26.4, OpenCV 4.8.1
   Solution: Restart kernel + run Cell 1 to fix versions

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22478MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaug

## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)


# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [22]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions
import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [23]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [ ]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset2"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset2"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset


In [25]:
!ls -a

.
..
.bashrc
.cache
.config
.docker
.gemini
.ipynb_checkpoints
.ipython
.jupyter
.jupyter_ystore.db
.jupyter_ystore.db-journal
.lastactive
.local
.npm
.nv
Untitled.ipynb
ce712-yolov8.ipynb
heatmap_1_hurricane-michael_00000304_post_disaster.png
heatmap_2_socal-fire_00001054_post_disaster.png
heatmap_3_hurricane-michael_00000500_post_disaster.png
notebook_template.ipynb
runs
visual-aid
xbd-dataset
xbd-dataset.zip
xview2-challenge-dataset-train-and-test
xview2-challenge-dataset-train-and-test.zip
yolo_dataset
yolov8n.pt


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [26]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [ ]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [ ]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [09:48<00:00, 11.54it/s]



Processing val split...


Converting val: 100%|██████████| 1699/1699 [02:30<00:00, 11.30it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3962
  Class distribution:
    No-Damage (class 0): 162251
    Moderate-Damage (class 1): 33263
    Total-Destruction (class 2): 13611

VAL:
  Total images: 1699
  Images with annotations: 984
  Class distribution:
    No-Damage (class 0): 41008
    Moderate-Damage (class 1): 9287
    Total-Destruction (class 2): 4588

✅ Dataset conversion completed!


In [30]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


✅ Created data.yaml at: ./yolo_dataset/data.yaml

Configuration:
# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: /home/jupyter/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes



## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


🚀 TRAINING PREPARATION

📊 Environment Check (CRITICAL for debugging):
   NumPy: 1.26.4
   OpenCV: 4.8.1
   ✅ Versions compatible!

✅ Dataset configuration found: ./yolo_dataset/data.yaml

📁 Dataset Structure Check:
   Train images: ./yolo_dataset/images/train - ✅
   Val images: ./yolo_dataset/images/val - ✅
   Train labels: ./yolo_dataset/labels/train - ✅
   Val labels: ./yolo_dataset/labels/val - ✅
   Train images count: 4337
   Val images count: 1325

🤖 Initializing YOLOv26n model...
   ✅ Model loaded successfully

🚀 Starting training...
   Model: YOLOv26n
   Epochs: 10000
   Batch Size: 16
   Image Size: 640

⚠️  DEBUGGING INFO:
   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible
   Current: NumPy 1.26.4, OpenCV 4.8.1
   Solution: Restart kernel + run Cell 1 to fix versions

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22478MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaug

## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)


# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [22]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions
import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [23]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [ ]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset


In [25]:
!ls -a

.
..
.bashrc
.cache
.config
.docker
.gemini
.ipynb_checkpoints
.ipython
.jupyter
.jupyter_ystore.db
.jupyter_ystore.db-journal
.lastactive
.local
.npm
.nv
Untitled.ipynb
ce712-yolov8.ipynb
heatmap_1_hurricane-michael_00000304_post_disaster.png
heatmap_2_socal-fire_00001054_post_disaster.png
heatmap_3_hurricane-michael_00000500_post_disaster.png
notebook_template.ipynb
runs
visual-aid
xbd-dataset
xbd-dataset.zip
xview2-challenge-dataset-train-and-test
xview2-challenge-dataset-train-and-test.zip
yolo_dataset
yolov8n.pt


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [26]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [ ]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [ ]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [09:48<00:00, 11.54it/s]



Processing val split...


Converting val: 100%|██████████| 1699/1699 [02:30<00:00, 11.30it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3962
  Class distribution:
    No-Damage (class 0): 162251
    Moderate-Damage (class 1): 33263
    Total-Destruction (class 2): 13611

VAL:
  Total images: 1699
  Images with annotations: 984
  Class distribution:
    No-Damage (class 0): 41008
    Moderate-Damage (class 1): 9287
    Total-Destruction (class 2): 4588

✅ Dataset conversion completed!


In [ ]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)


# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [22]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions
import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [23]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [ ]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset


In [25]:
!ls -a

.
..
.bashrc
.cache
.config
.docker
.gemini
.ipynb_checkpoints
.ipython
.jupyter
.jupyter_ystore.db
.jupyter_ystore.db-journal
.lastactive
.local
.npm
.nv
Untitled.ipynb
ce712-yolov8.ipynb
heatmap_1_hurricane-michael_00000304_post_disaster.png
heatmap_2_socal-fire_00001054_post_disaster.png
heatmap_3_hurricane-michael_00000500_post_disaster.png
notebook_template.ipynb
runs
visual-aid
xbd-dataset
xbd-dataset.zip
xview2-challenge-dataset-train-and-test
xview2-challenge-dataset-train-and-test.zip
yolo_dataset
yolov8n.pt


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [26]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [ ]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [ ]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [09:48<00:00, 11.54it/s]



Processing val split...


Converting val: 100%|██████████| 1699/1699 [02:30<00:00, 11.30it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3962
  Class distribution:
    No-Damage (class 0): 162251
    Moderate-Damage (class 1): 33263
    Total-Destruction (class 2): 13611

VAL:
  Total images: 1699
  Images with annotations: 984
  Class distribution:
    No-Damage (class 0): 41008
    Moderate-Damage (class 1): 9287
    Total-Destruction (class 2): 4588

✅ Dataset conversion completed!


In [30]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


✅ Created data.yaml at: ./yolo_dataset/data.yaml

Configuration:
# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: /home/jupyter/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes



## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


🚀 TRAINING PREPARATION

📊 Environment Check (CRITICAL for debugging):
   NumPy: 1.26.4
   OpenCV: 4.8.1
   ✅ Versions compatible!

✅ Dataset configuration found: ./yolo_dataset/data.yaml

📁 Dataset Structure Check:
   Train images: ./yolo_dataset/images/train - ✅
   Val images: ./yolo_dataset/images/val - ✅
   Train labels: ./yolo_dataset/labels/train - ✅
   Val labels: ./yolo_dataset/labels/val - ✅
   Train images count: 4337
   Val images count: 1325

🤖 Initializing YOLOv26n model...
   ✅ Model loaded successfully

🚀 Starting training...
   Model: YOLOv26n
   Epochs: 10000
   Batch Size: 16
   Image Size: 640

⚠️  DEBUGGING INFO:
   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible
   Current: NumPy 1.26.4, OpenCV 4.8.1
   Solution: Restart kernel + run Cell 1 to fix versions

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22478MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaug

## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)


# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [22]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions
import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [23]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [ ]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset


In [25]:
!ls -a

.
..
.bashrc
.cache
.config
.docker
.gemini
.ipynb_checkpoints
.ipython
.jupyter
.jupyter_ystore.db
.jupyter_ystore.db-journal
.lastactive
.local
.npm
.nv
Untitled.ipynb
ce712-yolov8.ipynb
heatmap_1_hurricane-michael_00000304_post_disaster.png
heatmap_2_socal-fire_00001054_post_disaster.png
heatmap_3_hurricane-michael_00000500_post_disaster.png
notebook_template.ipynb
runs
visual-aid
xbd-dataset
xbd-dataset.zip
xview2-challenge-dataset-train-and-test
xview2-challenge-dataset-train-and-test.zip
yolo_dataset
yolov8n.pt


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [26]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [ ]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [ ]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [09:48<00:00, 11.54it/s]



Processing val split...


Converting val: 100%|██████████| 1699/1699 [02:30<00:00, 11.30it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3962
  Class distribution:
    No-Damage (class 0): 162251
    Moderate-Damage (class 1): 33263
    Total-Destruction (class 2): 13611

VAL:
  Total images: 1699
  Images with annotations: 984
  Class distribution:
    No-Damage (class 0): 41008
    Moderate-Damage (class 1): 9287
    Total-Destruction (class 2): 4588

✅ Dataset conversion completed!


In [30]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


✅ Created data.yaml at: ./yolo_dataset/data.yaml

Configuration:
# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: /home/jupyter/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes



## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


🚀 TRAINING PREPARATION

📊 Environment Check (CRITICAL for debugging):
   NumPy: 1.26.4
   OpenCV: 4.8.1
   ✅ Versions compatible!

✅ Dataset configuration found: ./yolo_dataset/data.yaml

📁 Dataset Structure Check:
   Train images: ./yolo_dataset/images/train - ✅
   Val images: ./yolo_dataset/images/val - ✅
   Train labels: ./yolo_dataset/labels/train - ✅
   Val labels: ./yolo_dataset/labels/val - ✅
   Train images count: 4337
   Val images count: 1325

🤖 Initializing YOLOv26n model...
   ✅ Model loaded successfully

🚀 Starting training...
   Model: YOLOv26n
   Epochs: 10000
   Batch Size: 16
   Image Size: 640

⚠️  DEBUGGING INFO:
   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible
   Current: NumPy 1.26.4, OpenCV 4.8.1
   Solution: Restart kernel + run Cell 1 to fix versions

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22478MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaug

## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)


# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [22]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions
import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [23]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [ ]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset


In [25]:
!ls -a

.
..
.bashrc
.cache
.config
.docker
.gemini
.ipynb_checkpoints
.ipython
.jupyter
.jupyter_ystore.db
.jupyter_ystore.db-journal
.lastactive
.local
.npm
.nv
Untitled.ipynb
ce712-yolov8.ipynb
heatmap_1_hurricane-michael_00000304_post_disaster.png
heatmap_2_socal-fire_00001054_post_disaster.png
heatmap_3_hurricane-michael_00000500_post_disaster.png
notebook_template.ipynb
runs
visual-aid
xbd-dataset
xbd-dataset.zip
xview2-challenge-dataset-train-and-test
xview2-challenge-dataset-train-and-test.zip
yolo_dataset
yolov8n.pt


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [26]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [ ]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [ ]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [09:48<00:00, 11.54it/s]



Processing val split...


Converting val: 100%|██████████| 1699/1699 [02:30<00:00, 11.30it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3962
  Class distribution:
    No-Damage (class 0): 162251
    Moderate-Damage (class 1): 33263
    Total-Destruction (class 2): 13611

VAL:
  Total images: 1699
  Images with annotations: 984
  Class distribution:
    No-Damage (class 0): 41008
    Moderate-Damage (class 1): 9287
    Total-Destruction (class 2): 4588

✅ Dataset conversion completed!


In [30]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


✅ Created data.yaml at: ./yolo_dataset/data.yaml

Configuration:
# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: /home/jupyter/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes



## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


🚀 TRAINING PREPARATION

📊 Environment Check (CRITICAL for debugging):
   NumPy: 1.26.4
   OpenCV: 4.8.1
   ✅ Versions compatible!

✅ Dataset configuration found: ./yolo_dataset/data.yaml

📁 Dataset Structure Check:
   Train images: ./yolo_dataset/images/train - ✅
   Val images: ./yolo_dataset/images/val - ✅
   Train labels: ./yolo_dataset/labels/train - ✅
   Val labels: ./yolo_dataset/labels/val - ✅
   Train images count: 4337
   Val images count: 1325

🤖 Initializing YOLOv26n model...
   ✅ Model loaded successfully

🚀 Starting training...
   Model: YOLOv26n
   Epochs: 10000
   Batch Size: 16
   Image Size: 640

⚠️  DEBUGGING INFO:
   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible
   Current: NumPy 1.26.4, OpenCV 4.8.1
   Solution: Restart kernel + run Cell 1 to fix versions

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22478MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaug

## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)


# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [22]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions
import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [23]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [ ]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset


In [25]:
!ls -a

.
..
.bashrc
.cache
.config
.docker
.gemini
.ipynb_checkpoints
.ipython
.jupyter
.jupyter_ystore.db
.jupyter_ystore.db-journal
.lastactive
.local
.npm
.nv
Untitled.ipynb
ce712-yolov8.ipynb
heatmap_1_hurricane-michael_00000304_post_disaster.png
heatmap_2_socal-fire_00001054_post_disaster.png
heatmap_3_hurricane-michael_00000500_post_disaster.png
notebook_template.ipynb
runs
visual-aid
xbd-dataset
xbd-dataset.zip
xview2-challenge-dataset-train-and-test
xview2-challenge-dataset-train-and-test.zip
yolo_dataset
yolov8n.pt


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [26]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [ ]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [ ]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [09:48<00:00, 11.54it/s]



Processing val split...


Converting val: 100%|██████████| 1699/1699 [02:30<00:00, 11.30it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3962
  Class distribution:
    No-Damage (class 0): 162251
    Moderate-Damage (class 1): 33263
    Total-Destruction (class 2): 13611

VAL:
  Total images: 1699
  Images with annotations: 984
  Class distribution:
    No-Damage (class 0): 41008
    Moderate-Damage (class 1): 9287
    Total-Destruction (class 2): 4588

✅ Dataset conversion completed!


In [30]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


✅ Created data.yaml at: ./yolo_dataset/data.yaml

Configuration:
# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: /home/jupyter/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes



## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


🚀 TRAINING PREPARATION

📊 Environment Check (CRITICAL for debugging):
   NumPy: 1.26.4
   OpenCV: 4.8.1
   ✅ Versions compatible!

✅ Dataset configuration found: ./yolo_dataset/data.yaml

📁 Dataset Structure Check:
   Train images: ./yolo_dataset/images/train - ✅
   Val images: ./yolo_dataset/images/val - ✅
   Train labels: ./yolo_dataset/labels/train - ✅
   Val labels: ./yolo_dataset/labels/val - ✅
   Train images count: 4337
   Val images count: 1325

🤖 Initializing YOLOv26n model...
   ✅ Model loaded successfully

🚀 Starting training...
   Model: YOLOv26n
   Epochs: 10000
   Batch Size: 16
   Image Size: 640

⚠️  DEBUGGING INFO:
   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible
   Current: NumPy 1.26.4, OpenCV 4.8.1
   Solution: Restart kernel + run Cell 1 to fix versions

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22478MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaug

## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)


# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [22]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions
import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [23]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [ ]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset


In [25]:
!ls -a

.
..
.bashrc
.cache
.config
.docker
.gemini
.ipynb_checkpoints
.ipython
.jupyter
.jupyter_ystore.db
.jupyter_ystore.db-journal
.lastactive
.local
.npm
.nv
Untitled.ipynb
ce712-yolov8.ipynb
heatmap_1_hurricane-michael_00000304_post_disaster.png
heatmap_2_socal-fire_00001054_post_disaster.png
heatmap_3_hurricane-michael_00000500_post_disaster.png
notebook_template.ipynb
runs
visual-aid
xbd-dataset
xbd-dataset.zip
xview2-challenge-dataset-train-and-test
xview2-challenge-dataset-train-and-test.zip
yolo_dataset
yolov8n.pt


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [26]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [ ]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [ ]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [09:48<00:00, 11.54it/s]



Processing val split...


Converting val: 100%|██████████| 1699/1699 [02:30<00:00, 11.30it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3962
  Class distribution:
    No-Damage (class 0): 162251
    Moderate-Damage (class 1): 33263
    Total-Destruction (class 2): 13611

VAL:
  Total images: 1699
  Images with annotations: 984
  Class distribution:
    No-Damage (class 0): 41008
    Moderate-Damage (class 1): 9287
    Total-Destruction (class 2): 4588

✅ Dataset conversion completed!


In [30]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


✅ Created data.yaml at: ./yolo_dataset/data.yaml

Configuration:
# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: /home/jupyter/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes



## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


🚀 TRAINING PREPARATION

📊 Environment Check (CRITICAL for debugging):
   NumPy: 1.26.4
   OpenCV: 4.8.1
   ✅ Versions compatible!

✅ Dataset configuration found: ./yolo_dataset/data.yaml

📁 Dataset Structure Check:
   Train images: ./yolo_dataset/images/train - ✅
   Val images: ./yolo_dataset/images/val - ✅
   Train labels: ./yolo_dataset/labels/train - ✅
   Val labels: ./yolo_dataset/labels/val - ✅
   Train images count: 4337
   Val images count: 1325

🤖 Initializing YOLOv26n model...
   ✅ Model loaded successfully

🚀 Starting training...
   Model: YOLOv26n
   Epochs: 10000
   Batch Size: 16
   Image Size: 640

⚠️  DEBUGGING INFO:
   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible
   Current: NumPy 1.26.4, OpenCV 4.8.1
   Solution: Restart kernel + run Cell 1 to fix versions

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22478MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaug

## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)


# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [22]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions
import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [23]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [ ]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset


In [25]:
!ls -a

.
..
.bashrc
.cache
.config
.docker
.gemini
.ipynb_checkpoints
.ipython
.jupyter
.jupyter_ystore.db
.jupyter_ystore.db-journal
.lastactive
.local
.npm
.nv
Untitled.ipynb
ce712-yolov8.ipynb
heatmap_1_hurricane-michael_00000304_post_disaster.png
heatmap_2_socal-fire_00001054_post_disaster.png
heatmap_3_hurricane-michael_00000500_post_disaster.png
notebook_template.ipynb
runs
visual-aid
xbd-dataset
xbd-dataset.zip
xview2-challenge-dataset-train-and-test
xview2-challenge-dataset-train-and-test.zip
yolo_dataset
yolov8n.pt


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [26]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [ ]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [ ]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [09:48<00:00, 11.54it/s]



Processing val split...


Converting val: 100%|██████████| 1699/1699 [02:30<00:00, 11.30it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3962
  Class distribution:
    No-Damage (class 0): 162251
    Moderate-Damage (class 1): 33263
    Total-Destruction (class 2): 13611

VAL:
  Total images: 1699
  Images with annotations: 984
  Class distribution:
    No-Damage (class 0): 41008
    Moderate-Damage (class 1): 9287
    Total-Destruction (class 2): 4588

✅ Dataset conversion completed!


In [30]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


✅ Created data.yaml at: ./yolo_dataset/data.yaml

Configuration:
# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: /home/jupyter/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes



## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


🚀 TRAINING PREPARATION

📊 Environment Check (CRITICAL for debugging):
   NumPy: 1.26.4
   OpenCV: 4.8.1
   ✅ Versions compatible!

✅ Dataset configuration found: ./yolo_dataset/data.yaml

📁 Dataset Structure Check:
   Train images: ./yolo_dataset/images/train - ✅
   Val images: ./yolo_dataset/images/val - ✅
   Train labels: ./yolo_dataset/labels/train - ✅
   Val labels: ./yolo_dataset/labels/val - ✅
   Train images count: 4337
   Val images count: 1325

🤖 Initializing YOLOv26n model...
   ✅ Model loaded successfully

🚀 Starting training...
   Model: YOLOv26n
   Epochs: 10000
   Batch Size: 16
   Image Size: 640

⚠️  DEBUGGING INFO:
   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible
   Current: NumPy 1.26.4, OpenCV 4.8.1
   Solution: Restart kernel + run Cell 1 to fix versions

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22478MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaug

## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)


# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [22]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions
import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [23]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [ ]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset


In [25]:
!ls -a

.
..
.bashrc
.cache
.config
.docker
.gemini
.ipynb_checkpoints
.ipython
.jupyter
.jupyter_ystore.db
.jupyter_ystore.db-journal
.lastactive
.local
.npm
.nv
Untitled.ipynb
ce712-yolov8.ipynb
heatmap_1_hurricane-michael_00000304_post_disaster.png
heatmap_2_socal-fire_00001054_post_disaster.png
heatmap_3_hurricane-michael_00000500_post_disaster.png
notebook_template.ipynb
runs
visual-aid
xbd-dataset
xbd-dataset.zip
xview2-challenge-dataset-train-and-test
xview2-challenge-dataset-train-and-test.zip
yolo_dataset
yolov8n.pt


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [26]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [ ]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [ ]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [09:48<00:00, 11.54it/s]



Processing val split...


Converting val: 100%|██████████| 1699/1699 [02:30<00:00, 11.30it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3962
  Class distribution:
    No-Damage (class 0): 162251
    Moderate-Damage (class 1): 33263
    Total-Destruction (class 2): 13611

VAL:
  Total images: 1699
  Images with annotations: 984
  Class distribution:
    No-Damage (class 0): 41008
    Moderate-Damage (class 1): 9287
    Total-Destruction (class 2): 4588

✅ Dataset conversion completed!


In [30]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


✅ Created data.yaml at: ./yolo_dataset/data.yaml

Configuration:
# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: /home/jupyter/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes



## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


🚀 TRAINING PREPARATION

📊 Environment Check (CRITICAL for debugging):
   NumPy: 1.26.4
   OpenCV: 4.8.1
   ✅ Versions compatible!

✅ Dataset configuration found: ./yolo_dataset/data.yaml

📁 Dataset Structure Check:
   Train images: ./yolo_dataset/images/train - ✅
   Val images: ./yolo_dataset/images/val - ✅
   Train labels: ./yolo_dataset/labels/train - ✅
   Val labels: ./yolo_dataset/labels/val - ✅
   Train images count: 4337
   Val images count: 1325

🤖 Initializing YOLOv26n model...
   ✅ Model loaded successfully

🚀 Starting training...
   Model: YOLOv26n
   Epochs: 10000
   Batch Size: 16
   Image Size: 640

⚠️  DEBUGGING INFO:
   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible
   Current: NumPy 1.26.4, OpenCV 4.8.1
   Solution: Restart kernel + run Cell 1 to fix versions

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22478MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaug

## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)


# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [22]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions
import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [23]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [ ]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset


In [25]:
!ls -a

.
..
.bashrc
.cache
.config
.docker
.gemini
.ipynb_checkpoints
.ipython
.jupyter
.jupyter_ystore.db
.jupyter_ystore.db-journal
.lastactive
.local
.npm
.nv
Untitled.ipynb
ce712-yolov8.ipynb
heatmap_1_hurricane-michael_00000304_post_disaster.png
heatmap_2_socal-fire_00001054_post_disaster.png
heatmap_3_hurricane-michael_00000500_post_disaster.png
notebook_template.ipynb
runs
visual-aid
xbd-dataset
xbd-dataset.zip
xview2-challenge-dataset-train-and-test
xview2-challenge-dataset-train-and-test.zip
yolo_dataset
yolov8n.pt


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [26]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [ ]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [ ]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [09:48<00:00, 11.54it/s]



Processing val split...


Converting val: 100%|██████████| 1699/1699 [02:30<00:00, 11.30it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3962
  Class distribution:
    No-Damage (class 0): 162251
    Moderate-Damage (class 1): 33263
    Total-Destruction (class 2): 13611

VAL:
  Total images: 1699
  Images with annotations: 984
  Class distribution:
    No-Damage (class 0): 41008
    Moderate-Damage (class 1): 9287
    Total-Destruction (class 2): 4588

✅ Dataset conversion completed!


In [30]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


✅ Created data.yaml at: ./yolo_dataset/data.yaml

Configuration:
# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: /home/jupyter/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes



## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


🚀 TRAINING PREPARATION

📊 Environment Check (CRITICAL for debugging):
   NumPy: 1.26.4
   OpenCV: 4.8.1
   ✅ Versions compatible!

✅ Dataset configuration found: ./yolo_dataset/data.yaml

📁 Dataset Structure Check:
   Train images: ./yolo_dataset/images/train - ✅
   Val images: ./yolo_dataset/images/val - ✅
   Train labels: ./yolo_dataset/labels/train - ✅
   Val labels: ./yolo_dataset/labels/val - ✅
   Train images count: 4337
   Val images count: 1325

🤖 Initializing YOLOv26n model...
   ✅ Model loaded successfully

🚀 Starting training...
   Model: YOLOv26n
   Epochs: 10000
   Batch Size: 16
   Image Size: 640

⚠️  DEBUGGING INFO:
   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible
   Current: NumPy 1.26.4, OpenCV 4.8.1
   Solution: Restart kernel + run Cell 1 to fix versions

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22478MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaug

## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)


# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [22]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions
import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [23]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [ ]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset


In [25]:
!ls -a

.
..
.bashrc
.cache
.config
.docker
.gemini
.ipynb_checkpoints
.ipython
.jupyter
.jupyter_ystore.db
.jupyter_ystore.db-journal
.lastactive
.local
.npm
.nv
Untitled.ipynb
ce712-yolov8.ipynb
heatmap_1_hurricane-michael_00000304_post_disaster.png
heatmap_2_socal-fire_00001054_post_disaster.png
heatmap_3_hurricane-michael_00000500_post_disaster.png
notebook_template.ipynb
runs
visual-aid
xbd-dataset
xbd-dataset.zip
xview2-challenge-dataset-train-and-test
xview2-challenge-dataset-train-and-test.zip
yolo_dataset
yolov8n.pt


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [26]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [ ]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [ ]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [09:48<00:00, 11.54it/s]



Processing val split...


Converting val: 100%|██████████| 1699/1699 [02:30<00:00, 11.30it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3962
  Class distribution:
    No-Damage (class 0): 162251
    Moderate-Damage (class 1): 33263
    Total-Destruction (class 2): 13611

VAL:
  Total images: 1699
  Images with annotations: 984
  Class distribution:
    No-Damage (class 0): 41008
    Moderate-Damage (class 1): 9287
    Total-Destruction (class 2): 4588

✅ Dataset conversion completed!


In [30]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


✅ Created data.yaml at: ./yolo_dataset/data.yaml

Configuration:
# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: /home/jupyter/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes



## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


🚀 TRAINING PREPARATION

📊 Environment Check (CRITICAL for debugging):
   NumPy: 1.26.4
   OpenCV: 4.8.1
   ✅ Versions compatible!

✅ Dataset configuration found: ./yolo_dataset/data.yaml

📁 Dataset Structure Check:
   Train images: ./yolo_dataset/images/train - ✅
   Val images: ./yolo_dataset/images/val - ✅
   Train labels: ./yolo_dataset/labels/train - ✅
   Val labels: ./yolo_dataset/labels/val - ✅
   Train images count: 4337
   Val images count: 1325

🤖 Initializing YOLOv26n model...
   ✅ Model loaded successfully

🚀 Starting training...
   Model: YOLOv26n
   Epochs: 10000
   Batch Size: 16
   Image Size: 640

⚠️  DEBUGGING INFO:
   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible
   Current: NumPy 1.26.4, OpenCV 4.8.1
   Solution: Restart kernel + run Cell 1 to fix versions

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22478MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaug

## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)


# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [22]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions
import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [23]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [ ]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset


In [25]:
!ls -a

.
..
.bashrc
.cache
.config
.docker
.gemini
.ipynb_checkpoints
.ipython
.jupyter
.jupyter_ystore.db
.jupyter_ystore.db-journal
.lastactive
.local
.npm
.nv
Untitled.ipynb
ce712-yolov8.ipynb
heatmap_1_hurricane-michael_00000304_post_disaster.png
heatmap_2_socal-fire_00001054_post_disaster.png
heatmap_3_hurricane-michael_00000500_post_disaster.png
notebook_template.ipynb
runs
visual-aid
xbd-dataset
xbd-dataset.zip
xview2-challenge-dataset-train-and-test
xview2-challenge-dataset-train-and-test.zip
yolo_dataset
yolov8n.pt


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [26]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [ ]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [ ]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [09:48<00:00, 11.54it/s]



Processing val split...


Converting val: 100%|██████████| 1699/1699 [02:30<00:00, 11.30it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3962
  Class distribution:
    No-Damage (class 0): 162251
    Moderate-Damage (class 1): 33263
    Total-Destruction (class 2): 13611

VAL:
  Total images: 1699
  Images with annotations: 984
  Class distribution:
    No-Damage (class 0): 41008
    Moderate-Damage (class 1): 9287
    Total-Destruction (class 2): 4588

✅ Dataset conversion completed!


In [30]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


✅ Created data.yaml at: ./yolo_dataset/data.yaml

Configuration:
# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: /home/jupyter/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes



## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


🚀 TRAINING PREPARATION

📊 Environment Check (CRITICAL for debugging):
   NumPy: 1.26.4
   OpenCV: 4.8.1
   ✅ Versions compatible!

✅ Dataset configuration found: ./yolo_dataset/data.yaml

📁 Dataset Structure Check:
   Train images: ./yolo_dataset/images/train - ✅
   Val images: ./yolo_dataset/images/val - ✅
   Train labels: ./yolo_dataset/labels/train - ✅
   Val labels: ./yolo_dataset/labels/val - ✅
   Train images count: 4337
   Val images count: 1325

🤖 Initializing YOLOv26n model...
   ✅ Model loaded successfully

🚀 Starting training...
   Model: YOLOv26n
   Epochs: 10000
   Batch Size: 16
   Image Size: 640

⚠️  DEBUGGING INFO:
   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible
   Current: NumPy 1.26.4, OpenCV 4.8.1
   Solution: Restart kernel + run Cell 1 to fix versions

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22478MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaug

## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)


# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [22]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions
import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [23]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [ ]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset


In [25]:
!ls -a

.
..
.bashrc
.cache
.config
.docker
.gemini
.ipynb_checkpoints
.ipython
.jupyter
.jupyter_ystore.db
.jupyter_ystore.db-journal
.lastactive
.local
.npm
.nv
Untitled.ipynb
ce712-yolov8.ipynb
heatmap_1_hurricane-michael_00000304_post_disaster.png
heatmap_2_socal-fire_00001054_post_disaster.png
heatmap_3_hurricane-michael_00000500_post_disaster.png
notebook_template.ipynb
runs
visual-aid
xbd-dataset
xbd-dataset.zip
xview2-challenge-dataset-train-and-test
xview2-challenge-dataset-train-and-test.zip
yolo_dataset
yolov8n.pt


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [26]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [ ]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [ ]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [09:48<00:00, 11.54it/s]



Processing val split...


Converting val: 100%|██████████| 1699/1699 [02:30<00:00, 11.30it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3962
  Class distribution:
    No-Damage (class 0): 162251
    Moderate-Damage (class 1): 33263
    Total-Destruction (class 2): 13611

VAL:
  Total images: 1699
  Images with annotations: 984
  Class distribution:
    No-Damage (class 0): 41008
    Moderate-Damage (class 1): 9287
    Total-Destruction (class 2): 4588

✅ Dataset conversion completed!


In [30]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


✅ Created data.yaml at: ./yolo_dataset/data.yaml

Configuration:
# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: /home/jupyter/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes



## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


🚀 TRAINING PREPARATION

📊 Environment Check (CRITICAL for debugging):
   NumPy: 1.26.4
   OpenCV: 4.8.1
   ✅ Versions compatible!

✅ Dataset configuration found: ./yolo_dataset/data.yaml

📁 Dataset Structure Check:
   Train images: ./yolo_dataset/images/train - ✅
   Val images: ./yolo_dataset/images/val - ✅
   Train labels: ./yolo_dataset/labels/train - ✅
   Val labels: ./yolo_dataset/labels/val - ✅
   Train images count: 4337
   Val images count: 1325

🤖 Initializing YOLOv26n model...
   ✅ Model loaded successfully

🚀 Starting training...
   Model: YOLOv26n
   Epochs: 10000
   Batch Size: 16
   Image Size: 640

⚠️  DEBUGGING INFO:
   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible
   Current: NumPy 1.26.4, OpenCV 4.8.1
   Solution: Restart kernel + run Cell 1 to fix versions

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22478MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaug

## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)


# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [22]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions
import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [23]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [ ]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset


In [25]:
!ls -a

.
..
.bashrc
.cache
.config
.docker
.gemini
.ipynb_checkpoints
.ipython
.jupyter
.jupyter_ystore.db
.jupyter_ystore.db-journal
.lastactive
.local
.npm
.nv
Untitled.ipynb
ce712-yolov8.ipynb
heatmap_1_hurricane-michael_00000304_post_disaster.png
heatmap_2_socal-fire_00001054_post_disaster.png
heatmap_3_hurricane-michael_00000500_post_disaster.png
notebook_template.ipynb
runs
visual-aid
xbd-dataset
xbd-dataset.zip
xview2-challenge-dataset-train-and-test
xview2-challenge-dataset-train-and-test.zip
yolo_dataset
yolov8n.pt


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [26]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [ ]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [ ]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [09:48<00:00, 11.54it/s]



Processing val split...


Converting val: 100%|██████████| 1699/1699 [02:30<00:00, 11.30it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3962
  Class distribution:
    No-Damage (class 0): 162251
    Moderate-Damage (class 1): 33263
    Total-Destruction (class 2): 13611

VAL:
  Total images: 1699
  Images with annotations: 984
  Class distribution:
    No-Damage (class 0): 41008
    Moderate-Damage (class 1): 9287
    Total-Destruction (class 2): 4588

✅ Dataset conversion completed!


In [30]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


✅ Created data.yaml at: ./yolo_dataset/data.yaml

Configuration:
# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: /home/jupyter/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes



## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


🚀 TRAINING PREPARATION

📊 Environment Check (CRITICAL for debugging):
   NumPy: 1.26.4
   OpenCV: 4.8.1
   ✅ Versions compatible!

✅ Dataset configuration found: ./yolo_dataset/data.yaml

📁 Dataset Structure Check:
   Train images: ./yolo_dataset/images/train - ✅
   Val images: ./yolo_dataset/images/val - ✅
   Train labels: ./yolo_dataset/labels/train - ✅
   Val labels: ./yolo_dataset/labels/val - ✅
   Train images count: 4337
   Val images count: 1325

🤖 Initializing YOLOv26n model...
   ✅ Model loaded successfully

🚀 Starting training...
   Model: YOLOv26n
   Epochs: 10000
   Batch Size: 16
   Image Size: 640

⚠️  DEBUGGING INFO:
   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible
   Current: NumPy 1.26.4, OpenCV 4.8.1
   Solution: Restart kernel + run Cell 1 to fix versions

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22478MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaug

## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)


# Building Detection from Satellite Images using YOLOv26

This notebook implements the research paper: "Building Detection from Satellite Images using Deep Learning" by Yap et al.

## Overview
- **Model**: YOLOv26 for building detection and damage assessment
- **Dataset**: xView2/xBD Challenge Dataset
- **Task**: Detect buildings and classify damage levels (3 classes)
- **Classes**: 
  - No-Damage (class 0)
  - Moderate-Damage (class 1) 
  - Total-Destruction (class 2)

## Methodology
1. Parse JSON/GeoJSON annotations
2. Convert polygon annotations to YOLO bounding box format
3. Map 5 damage classes → 3 classes (as per research paper)
4. Create train/validation split (80/20)
5. Prepare YOLOv26 dataset structure
6. Train YOLOv26 model (50 epochs, batch size with accumulation)
7. Evaluate and visualize results


In [22]:
# Install required packages (pin numpy/scipy/sklearn for compatibility)
# CRITICAL ISSUE: NumPy 2.x + OpenCV 4.12.0 causes "imdecode" errors during training
# Root cause: OpenCV 4.12.0 requires NumPy 2.x, but we need NumPy 1.26.4 for compatibility
# Solution: Install ultralytics first, then force reinstall compatible versions
import sys
import subprocess

def get_package_version(package_name):
    """Get installed version of a package - for debugging"""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "Not found"

print("="*80)
print("🔧 PACKAGE INSTALLATION AND VERSION FIX")
print("="*80)

print("\n📦 Step 1: Upgrading pip...")
!{sys.executable} -m pip install --upgrade pip -q

print("\n📦 Step 2: Checking initial package versions (for debugging)...")
numpy_init = get_package_version('numpy')
opencv_init = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_init}")
print(f"   OpenCV: {opencv_init}")

print("\n📦 Step 3: Uninstalling conflicting packages...")
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn opencv-python opencv-python-headless -q

print("\n📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...")
!{sys.executable} -m pip install ultralytics --no-cache-dir -q

print("\n📦 Step 5: Checking versions after ultralytics (for debugging)...")
numpy_after = get_package_version('numpy')
opencv_after = get_package_version('opencv-python-headless')
print(f"   NumPy: {numpy_after}")
print(f"   OpenCV: {opencv_after}")

print("\n📦 Step 6: Force reinstalling compatible NumPy 1.26.4...")
# CRITICAL: Use --no-deps to prevent pip from upgrading numpy
!{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scipy==1.11.4 --no-cache-dir -q
!{sys.executable} -m pip install --force-reinstall scikit-learn==1.3.2 --no-cache-dir -q

print("\n📦 Step 7: Force reinstalling compatible OpenCV 4.8.1.78...")
# CRITICAL: OpenCV 4.12.0 requires NumPy 2.x, so we MUST use 4.8.1.78
# Use --no-deps to prevent OpenCV from pulling NumPy 2.x
!{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q

print("\n📦 Step 8: Installing other dependencies...")
!{sys.executable} -m pip install shapely --no-cache-dir -q
!{sys.executable} -m pip install geopandas --no-cache-dir -q

print("\n" + "="*80)
print("🔍 VERSION VERIFICATION (CRITICAL)")
print("="*80)

# Import and verify versions
import numpy as np
import cv2

numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Installed Versions:")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

# Check NumPy
numpy_ok = False
if numpy_version.startswith('2.'):
    print("\n❌ ERROR: NumPy 2.x detected! This WILL cause OpenCV imdecode errors.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps numpy==1.26.4 --no-cache-dir -q
    import importlib
    importlib.reload(np)
    numpy_version = np.__version__
    print(f"   After fix attempt: {numpy_version}")
    if numpy_version.startswith('2.'):
        print("   ❌ FAILED to fix! Please restart kernel and run this cell again.")
        raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")
    else:
        numpy_ok = True
elif numpy_version.startswith('1.26'):
    numpy_ok = True
    print("   ✅ NumPy version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected NumPy version {numpy_version}")

# Check OpenCV
opencv_ok = False
if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ ERROR: OpenCV {opencv_version} detected! This requires NumPy 2.x.")
    print("   Attempting emergency fix...")
    !{sys.executable} -m pip install --force-reinstall --no-deps opencv-python-headless==4.8.1.78 --no-cache-dir -q
    import importlib
    importlib.reload(cv2)
    opencv_version = cv2.__version__
    print(f"   After fix attempt: {opencv_version}")
    if not opencv_version.startswith('4.8'):
        print("   ❌ FAILED to fix! Please restart kernel.")
        raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")
    else:
        opencv_ok = True
elif opencv_version.startswith('4.8'):
    opencv_ok = True
    print("   ✅ OpenCV version OK")
else:
    print(f"   ⚠️ WARNING: Unexpected OpenCV version {opencv_version}")

# Final status
print("\n" + "="*80)
if numpy_ok and opencv_ok:
    print("✅ ALL VERSIONS CORRECT! Ready for training.")
    print(f"   Compatible stack: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("⚠️ VERSION WARNINGS DETECTED - Training may fail!")
    print(f"   NumPy OK: {numpy_ok}, OpenCV OK: {opencv_ok}")
print("="*80)


🔧 PACKAGE INSTALLATION AND VERSION FIX

📦 Step 1: Upgrading pip...

📦 Step 2: Checking initial package versions (for debugging)...
   NumPy: 1.26.4
   OpenCV: 4.8.1.78

📦 Step 3: Uninstalling conflicting packages...

📦 Step 4: Installing ultralytics (may temporarily upgrade numpy/opencv)...

📦 Step 5: Checking versions after ultralytics (for debugging)...
   NumPy: 2.2.6
   OpenCV: Not found

📦 Step 6: Force reinstalling compatible NumPy 1.26.4...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [23]:
# Import necessary libraries
# Note: numpy should already be set to 1.26.4 from previous cell
import os
import json
import shutil
import numpy as np
print(f"NumPy version check: {np.__version__}")  # Verify numpy version

import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import random
from shapely.geometry import Polygon, box
from shapely.wkt import loads as wkt_loads
import warnings
warnings.filterwarnings('ignore')

# YOLOv26
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


NumPy version check: 1.26.4
Libraries imported successfully!


## Step 1: Dataset Configuration


In [ ]:
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
# Dataset configuration
DATASET_BASE = "xbd-dataset/xbd/"

# Check if path exists, try alternatives
if not os.path.exists(DATASET_BASE):
    alternatives = [
        "/kaggle/input/xview2-challenge-dataset-tier-3-data",
        "./xview2_dataset",
        "../xview2_dataset"
    ]
    for alt_path in alternatives:
        if os.path.exists(alt_path):
            DATASET_BASE = alt_path
            print(f"Using alternative path: {DATASET_BASE}")
            break
    else:
        print("⚠️ Dataset path not found. Please update DATASET_BASE.")
        DATASET_BASE = None

if DATASET_BASE:
    print(f"✅ Dataset base path: {DATASET_BASE}")

    # Define paths
    TRAIN_IMAGES_DIR = os.path.join(DATASET_BASE, "train/images")
    TRAIN_LABELS_DIR = os.path.join(DATASET_BASE, "train/labels")
    TEST_IMAGES_DIR = os.path.join(DATASET_BASE, "test/images")

    # Output directory for YOLO format
    OUTPUT_DIR = "./yolo_dataset"
    YOLO_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
    YOLO_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")

    print(f"Train images: {TRAIN_IMAGES_DIR}")
    print(f"Train labels: {TRAIN_LABELS_DIR}")
    print(f"Test images: {TEST_IMAGES_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")


✅ Dataset base path: xbd-dataset/xbd/
Train images: xbd-dataset/xbd/train/images
Train labels: xbd-dataset/xbd/train/labels
Test images: xbd-dataset/xbd/test/images
Output directory: ./yolo_dataset


In [25]:
!ls -a

.
..
.bashrc
.cache
.config
.docker
.gemini
.ipynb_checkpoints
.ipython
.jupyter
.jupyter_ystore.db
.jupyter_ystore.db-journal
.lastactive
.local
.npm
.nv
Untitled.ipynb
ce712-yolov8.ipynb
heatmap_1_hurricane-michael_00000304_post_disaster.png
heatmap_2_socal-fire_00001054_post_disaster.png
heatmap_3_hurricane-michael_00000500_post_disaster.png
notebook_template.ipynb
runs
visual-aid
xbd-dataset
xbd-dataset.zip
xview2-challenge-dataset-train-and-test
xview2-challenge-dataset-train-and-test.zip
yolo_dataset
yolov8n.pt


## Step 2: Damage Class Mapping

According to the research paper, we need to map the original 5 damage classes to 3 classes:
- **No-Damage**: Class 1 (unchanged)
- **Moderate-Damage**: Classes 2-3 combined
- **Total-Destruction**: Class 4


In [26]:
# Damage class mapping (from paper: 5 classes → 3 classes)
# Original xView2 classes:
# 0: Background (not used in YOLO)
# 1: No-Damage
# 2: Minor-Damage
# 3: Major-Damage
# 4: Destroyed

# Mapping to 3 classes for YOLO:
CLASS_MAPPING = {
    1: 0,  # No-Damage → class 0
    2: 1,  # Minor-Damage → Moderate-Damage (class 1)
    3: 1,  # Major-Damage → Moderate-Damage (class 1)
    4: 2,  # Destroyed → Total-Destruction (class 2)
}

CLASS_NAMES = {
    0: "No-Damage",
    1: "Moderate-Damage",
    2: "Total-Destruction"
}

print("Class Mapping:")
for orig, new in CLASS_MAPPING.items():
    print(f"  Original class {orig} → YOLO class {new} ({CLASS_NAMES[new]})")


Class Mapping:
  Original class 1 → YOLO class 0 (No-Damage)
  Original class 2 → YOLO class 1 (Moderate-Damage)
  Original class 3 → YOLO class 1 (Moderate-Damage)
  Original class 4 → YOLO class 2 (Total-Destruction)


## Step 3: Annotation Parsing and Conversion Functions


In [ ]:
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")
from shapely.errors import WKTReadingError

def parse_wkt_polygon(wkt_string, verbose=False):
    """Parse WKT polygon string to get coordinates (polygon or multipolygon)."""
    if not wkt_string:
        return None

    def _sanitize_coords(coords):
        if coords is None or len(coords) == 0:
            return None
        # Remove duplicate closing point if present
        if len(coords) > 1 and coords[0] == coords[-1]:
            coords = coords[:-1]
        return coords if len(coords) >= 3 else None

    # Try shapely parsing first
    geom = None
    try:
        geom = wkt_loads(wkt_string)
    except (WKTReadingError, Exception) as e:
        if verbose:
            print(f"[WKT error] {e} :: {wkt_string[:120]}...")
        geom = None

    coords = None
    if geom is not None:
        try:
            if geom.geom_type == "Polygon":
                # Extract coordinates more robustly
                try:
                    # Try direct conversion
                    coords = list(geom.exterior.coords)
                except (AttributeError, TypeError, ValueError) as e:
                    # Fallback: use numpy array conversion
                    try:
                        import numpy as np
                        coords_array = np.array(geom.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        # Last resort: try accessing xy attribute
                        try:
                            if hasattr(geom.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(geom.exterior.xy[0], geom.exterior.xy[1])]
                            else:
                                # Final fallback: iterate through coordinate sequence
                                coords = [(float(coord[0]), float(coord[1])) for coord in geom.exterior.coords]
                        except Exception:
                            coords = None
            elif geom.geom_type == "MultiPolygon" and len(geom.geoms) > 0:
                # Pick the largest polygon
                largest = max(geom.geoms, key=lambda g: g.area)
                try:
                    coords = list(largest.exterior.coords)
                except (AttributeError, TypeError, ValueError):
                    try:
                        import numpy as np
                        coords_array = np.array(largest.exterior.coords)
                        coords = [tuple(coord) for coord in coords_array]
                    except Exception:
                        try:
                            if hasattr(largest.exterior, 'xy'):
                                coords = [(float(x), float(y)) for x, y in zip(largest.exterior.xy[0], largest.exterior.xy[1])]
                            else:
                                coords = [(float(coord[0]), float(coord[1])) for coord in largest.exterior.coords]
                        except Exception:
                            coords = None
            else:
                # Unsupported geometry
                coords = None
        except Exception as e:
            if verbose:
                print(f"[WKT coordinate error] {e} :: {wkt_string[:120]}...")
            coords = None

        coords = _sanitize_coords(coords) if coords else None

    # Fallback manual parser if shapely failed
    if coords is None:
        try:
            raw = wkt_string.strip()
            # Handle POLYGON format: POLYGON ((x1 y1, x2 y2, ...))
            if raw.upper().startswith("POLYGON"):
                # Remove "POLYGON" prefix
                raw = raw[7:].strip()
                # Remove outer parentheses
                if raw.startswith("((") and raw.endswith("))"):
                    raw = raw[2:-2]
                elif raw.startswith("(") and raw.endswith(")"):
                    raw = raw[1:-1]
                # Split by comma and parse coordinates
                parts = [p.strip() for p in raw.split(",") if p.strip()]
                manual_coords = []
                for part in parts:
                    xy = part.split()
                    if len(xy) >= 2:
                        try:
                            manual_coords.append((float(xy[0]), float(xy[1])))
                        except ValueError:
                            continue
                coords = _sanitize_coords(manual_coords)
            # Handle MULTIPOLYGON format
            elif raw.upper().startswith("MULTIPOLYGON"):
                # Extract first polygon from multipolygon
                raw = raw[12:].strip()
                if raw.startswith("(("):
                    # Find the matching closing parentheses for first polygon
                    depth = 0
                    end_idx = 0
                    for i, char in enumerate(raw):
                        if char == '(':
                            depth += 1
                        elif char == ')':
                            depth -= 1
                            if depth == 0:
                                end_idx = i + 1
                                break
                    raw = raw[2:end_idx-1]  # Remove outer parentheses
                    parts = [p.strip() for p in raw.split(",") if p.strip()]
                    manual_coords = []
                    for part in parts:
                        xy = part.split()
                        if len(xy) >= 2:
                            try:
                                manual_coords.append((float(xy[0]), float(xy[1])))
                            except ValueError:
                                continue
                    coords = _sanitize_coords(manual_coords)
        except Exception as e:
            if verbose:
                print(f"[Fallback WKT parsing failed] {e} :: {wkt_string[:120]}...")
            coords = None

    return coords

def polygon_to_bbox(polygon_coords):
    """Convert polygon coordinates to bounding box [x_min, y_min, x_max, y_max]"""
    if not polygon_coords or len(polygon_coords) == 0:
        return None

    x_coords = [coord[0] for coord in polygon_coords]
    y_coords = [coord[1] for coord in polygon_coords]

    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)

    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box to YOLO format (normalized center_x, center_y, width, height)"""
    x_min, y_min, x_max, y_max = bbox

    # Clamp coordinates to image bounds
    x_min = max(0, min(x_min, img_width))
    x_max = max(0, min(x_max, img_width))
    y_min = max(0, min(y_min, img_height))
    y_max = max(0, min(y_max, img_height))

    # Ensure valid bbox
    if x_max <= x_min or y_max <= y_min:
        return None

    # Calculate center and dimensions
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    # Normalize
    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    # Validate normalized coordinates
    if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None

    return [center_x, center_y, width, height]

def parse_annotation_file(json_path):
    """Parse xView2 annotation JSON file and extract building annotations"""
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        annotations = []

        # xView2 format has 'features' as a dict with 'xy' (pixel coordinates) and 'lng_lat' (geographic)
        # We use 'xy' for image coordinates
        if 'features' in data:
            features_dict = data['features']

            # Check if 'xy' key exists (pixel coordinates)
            if isinstance(features_dict, dict) and 'xy' in features_dict:
                features = features_dict['xy']
            elif isinstance(features_dict, list):
                # Sometimes features is directly a list
                features = features_dict
            else:
                return annotations

            for feature in features:
                if not isinstance(feature, dict):
                    continue

                # Skip if not a building feature (optional check)
                if 'properties' in feature:
                    props = feature['properties']
                    # Only process building features (skip if feature_type exists and is not 'building')
                    if 'feature_type' in props and props.get('feature_type') != 'building':
                        continue

                if 'properties' in feature and 'wkt' in feature:
                    # Get damage class from properties
                    properties = feature['properties']

                    # Damage class extraction - try multiple fields
                    damage_class = None

                    # Method 1: Check 'subtype' field (most common in xView2)
                    if 'subtype' in properties:
                        subtype = str(properties['subtype']).lower().strip()
                        # Match damage types - be flexible with naming
                        if 'no-damage' in subtype or 'no_damage' in subtype or subtype == '1' or subtype == 'no damage':
                            damage_class = 1
                        elif 'minor-damage' in subtype or 'minor_damage' in subtype or subtype == '2' or subtype == 'minor damage':
                            damage_class = 2
                        elif 'major-damage' in subtype or 'major_damage' in subtype or subtype == '3' or subtype == 'major damage':
                            damage_class = 3
                        elif 'destroyed' in subtype or subtype == '4' or 'destroy' in subtype:
                            damage_class = 4

                    # Method 2: Check 'damage' field (numeric)
                    if damage_class is None and 'damage' in properties:
                        try:
                            damage_val = int(properties['damage'])
                            if 1 <= damage_val <= 4:
                                damage_class = damage_val
                        except:
                            pass

                    # Method 3: Check 'type' field
                    if damage_class is None and 'type' in properties:
                        type_val = str(properties['type']).lower()
                        if 'no' in type_val and 'damage' in type_val:
                            damage_class = 1
                        elif 'minor' in type_val:
                            damage_class = 2
                        elif 'major' in type_val:
                            damage_class = 3
                        elif 'destroyed' in type_val or 'destroy' in type_val:
                            damage_class = 4

                    # Parse polygon from WKT
                    wkt_string = feature.get('wkt', '')
                    if not wkt_string:
                        continue

                    polygon_coords = parse_wkt_polygon(wkt_string)

                    # Debug: if we have damage_class but no coords, there's a parsing issue
                    if damage_class and not polygon_coords:
                        # WKT parsing failed - skip this feature
                        continue

                    if polygon_coords and damage_class and damage_class in CLASS_MAPPING:
                        bbox = polygon_to_bbox(polygon_coords)
                        if bbox and len(bbox) == 4:
                            # Validate bbox has valid dimensions
                            x_min, y_min, x_max, y_max = bbox
                            if x_max > x_min and y_max > y_min:
                                annotations.append({
                                    'bbox': bbox,
                                    'damage_class': damage_class,
                                    'yolo_class': CLASS_MAPPING[damage_class]
                                })

        return annotations

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return []

# Test the parsing function
print("✅ Annotation parsing functions defined")


✅ Annotation parsing functions defined


## Step 3.5: Debug Annotation Parsing

Let's debug why annotations aren't being found


In [ ]:
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")
# Debug: Check JSON files and test parsing
if DATASET_BASE and os.path.exists(DATASET_BASE):
    labels_dir = os.path.join(DATASET_BASE, "train/labels")
    images_dir = os.path.join(DATASET_BASE, "train/images")

    if os.path.exists(labels_dir):
        # List some JSON files
        json_files = [f for f in os.listdir(labels_dir) if f.endswith('.json')]
        print(f"Found {len(json_files)} JSON files in labels directory")
        print(f"\nSample JSON files (first 5):")
        for f in json_files[:5]:
            print(f"  - {f}")

        # List some image files
        if os.path.exists(images_dir):
            img_files = [f for f in os.listdir(images_dir) if 'post' in f.lower() and f.endswith('.png')]
            print(f"\nFound {len(img_files)} post-disaster image files")
            print(f"\nSample image files (first 5):")
            for f in img_files[:5]:
                print(f"  - {f}")

            # Test matching logic
            print(f"\n" + "="*80)
            print("TESTING FILE MATCHING")
            print("="*80)
            test_img = img_files[0] if img_files else None
            if test_img:
                print(f"\nTest image: {test_img}")
                base_name = test_img.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                print(f"Extracted base_name: {base_name}")
                expected_json = f"{base_name}_post_disaster.json"
                print(f"Expected JSON: {expected_json}")

                json_path = os.path.join(labels_dir, expected_json)
                print(f"JSON path exists: {os.path.exists(json_path)}")

                if os.path.exists(json_path):
                    print(f"\n✅ Found matching JSON! Testing parsing...")

                    # Load and inspect JSON first
                    with open(json_path, 'r') as f:
                        test_data = json.load(f)

                    if 'features' in test_data and 'xy' in test_data['features']:
                        xy_features = test_data['features']['xy']
                        print(f"Found {len(xy_features)} xy features")

                        # Test parsing each feature manually
                        for i, feat in enumerate(xy_features[:3]):  # Test first 3
                            if 'properties' in feat and 'wkt' in feat:
                                props = feat['properties']
                                wkt = feat['wkt']

                                print(f"\n  Feature {i+1}:")
                                print(f"    Feature type: {props.get('feature_type', 'N/A')}")
                                print(f"    Subtype: {props.get('subtype', 'N/A')}")

                                # Test WKT parsing
                                coords = parse_wkt_polygon(wkt)
                                print(f"    WKT parsed: {len(coords) if coords else 0} coordinates")

                                if coords:
                                    bbox = polygon_to_bbox(coords)
                                    print(f"    Bbox: {bbox}")

                                    # Test damage class extraction
                                    subtype = str(props.get('subtype', '')).lower()
                                    damage_class = None
                                    if 'major-damage' in subtype:
                                        damage_class = 3
                                    elif 'minor-damage' in subtype:
                                        damage_class = 2
                                    elif 'no-damage' in subtype:
                                        damage_class = 1
                                    elif 'destroyed' in subtype:
                                        damage_class = 4

                                    print(f"    Extracted damage_class: {damage_class}")
                                    if damage_class and damage_class in CLASS_MAPPING:
                                        print(f"    → YOLO class: {CLASS_MAPPING[damage_class]} ({CLASS_NAMES[CLASS_MAPPING[damage_class]]})")

                    # Now test the actual parsing function
                    annotations = parse_annotation_file(json_path)
                    print(f"\n📊 parse_annotation_file() returned {len(annotations)} annotations")

                    if len(annotations) > 0:
                        print(f"\nSample annotation:")
                        print(f"  Damage class: {annotations[0]['damage_class']}")
                        print(f"  YOLO class: {annotations[0]['yolo_class']}")
                        print(f"  Bbox: {annotations[0]['bbox']}")
                    else:
                        print("\n❌ parse_annotation_file() returned 0 annotations - checking why...")
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                        print("\n⚠️ No annotations extracted. Inspecting JSON structure...")
                        print(f"JSON keys: {list(data.keys())}")
                        if 'features' in data:
                            print(f"Features type: {type(data['features'])}")
                            if isinstance(data['features'], dict):
                                print(f"Features dict keys: {list(data['features'].keys())}")
                                if 'xy' in data['features']:
                                    print(f"Number of xy features: {len(data['features']['xy'])}")
                                    if len(data['features']['xy']) > 0:
                                        feature = data['features']['xy'][0]
                                        print(f"First feature keys: {list(feature.keys())}")
                                        if 'properties' in feature:
                                            print(f"Properties: {feature['properties']}")
                                        if 'wkt' in feature:
                                            wkt_str = feature['wkt']
                                            print(f"WKT string (first 100 chars): {wkt_str[:100]}")
                                            # Test WKT parsing
                                            coords = parse_wkt_polygon(wkt_str)
                                            print(f"Parsed coordinates: {len(coords) if coords else 0} points")
                                            if coords:
                                                bbox = polygon_to_bbox(coords)
                                                print(f"Bounding box: {bbox}")
                                                # Check damage class extraction
                                                props = feature['properties']
                                                if 'subtype' in props:
                                                    subtype = str(props['subtype']).lower()
                                                    print(f"Subtype: '{subtype}'")
                                                    # Test matching
                                                    if 'major-damage' in subtype:
                                                        print("✅ Would match major-damage → class 3")
                                                    elif 'minor-damage' in subtype:
                                                        print("✅ Would match minor-damage → class 2")
                                                    elif 'no-damage' in subtype:
                                                        print("✅ Would match no-damage → class 1")
                                                    elif 'destroyed' in subtype:
                                                        print("✅ Would match destroyed → class 4")
                                                    else:
                                                        print(f"⚠️ Subtype '{subtype}' doesn't match any pattern")
                else:
                    # Try to find similar JSON files
                    print(f"\n⚠️ Exact match not found. Searching for similar files...")
                    similar = [f for f in json_files if base_name in f]
                    print(f"Found {len(similar)} similar JSON files:")
                    for f in similar[:3]:
                        print(f"  - {f}")
    else:
        print(f"⚠️ Labels directory not found: {labels_dir}")
else:
    print("⚠️ Dataset base not found")


Found 18336 JSON files in labels directory

Sample JSON files (first 5):
  - woolsey-fire_00000368_post_disaster.json
  - joplin-tornado_00000079_post_disaster.json
  - nepal-flooding_00000505_post_disaster.json
  - sunda-tsunami_00000051_pre_disaster.json
  - nepal-flooding_00000477_pre_disaster.json

Found 8494 post-disaster image files

Sample image files (first 5):
  - socal-fire_00000812_post_disaster.png
  - hurricane-michael_00000304_post_disaster.png
  - pinery-bushfire_00000798_post_disaster.png
  - woolsey-fire_00000348_post_disaster.png
  - nepal-flooding_00000522_post_disaster.png

TESTING FILE MATCHING

Test image: socal-fire_00000812_post_disaster.png
Extracted base_name: socal-fire_00000812
Expected JSON: socal-fire_00000812_post_disaster.json
JSON path exists: True

✅ Found matching JSON! Testing parsing...
Found 0 xy features

📊 parse_annotation_file() returned 0 annotations

❌ parse_annotation_file() returned 0 annotations - checking why...

⚠️ No annotations extracte

In [ ]:
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")
def validate_image(image_path):
    """Validate that an image can be read by both PIL and OpenCV"""
    try:
        # Try PIL
        img = Image.open(image_path)
        img.verify()
        img = Image.open(image_path)  # Reopen after verify
        img.load()  # Load image data
        img.close()

        # Try OpenCV
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            return False

        return True
    except Exception:
        return False

def convert_to_yolo_format(dataset_base, output_dir, split='train', train_ratio=0.8):
    """
    Convert xView2 dataset to YOLO format

    Args:
        dataset_base: Base path to xView2 dataset
        output_dir: Output directory for YOLO dataset
        split: 'train' or 'test'
        train_ratio: Ratio for train/val split (default 0.8 for 80/20)
    """

    if split == 'train':
        images_dir = os.path.join(dataset_base, "train/images")
        labels_dir = os.path.join(dataset_base, "train/labels")
    else:
        images_dir = os.path.join(dataset_base, "test/images")
        labels_dir = None

    # Create output directories
    train_images_dir = os.path.join(output_dir, "images", "train")
    val_images_dir = os.path.join(output_dir, "images", "val")
    train_labels_dir = os.path.join(output_dir, "labels", "train")
    val_labels_dir = os.path.join(output_dir, "labels", "val")

    for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        os.makedirs(dir_path, exist_ok=True)

    # Get all image files
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Found {len(image_files)} image files")

    # Filter for post-disaster images (we'll use post-disaster for damage assessment)
    post_disaster_files = [f for f in image_files if 'post' in f.lower()]
    print(f"Found {len(post_disaster_files)} post-disaster images")

    # Shuffle and split
    random.shuffle(post_disaster_files)
    split_idx = int(len(post_disaster_files) * train_ratio)
    train_files = post_disaster_files[:split_idx]
    val_files = post_disaster_files[split_idx:]

    print(f"Train: {len(train_files)}, Val: {len(val_files)}")

    # Process files
    stats = {
        'train': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()},
        'val': {'total': 0, 'with_annotations': 0, 'class_counts': Counter()}
    }

    for split_name, files in [('train', train_files), ('val', val_files)]:
        print(f"\nProcessing {split_name} split...")

        for img_file in tqdm(files, desc=f"Converting {split_name}"):
            stats[split_name]['total'] += 1

            # Get image path
            img_path = os.path.join(images_dir, img_file)

            # Get corresponding label file (for train split)
            annotations = []
            if labels_dir and split == 'train':
                # Find corresponding JSON file - try multiple naming patterns
                base_name = img_file.replace('_post_disaster.png', '').replace('_pre_disaster.png', '')
                base_name = base_name.replace('.png', '').replace('.jpg', '').replace('.jpeg', '')

                # Try different JSON file naming patterns
                json_candidates = [
                    f"{base_name}_post_disaster.json",
                    f"{base_name}_post.json",
                    f"{base_name}.json",
                    img_file.replace('.png', '.json').replace('.jpg', '.json').replace('.jpeg', '.json')
                ]

                json_path = None
                for candidate in json_candidates:
                    candidate_path = os.path.join(labels_dir, candidate)
                    if os.path.exists(candidate_path):
                        json_path = candidate_path
                        break

                if json_path:
                    annotations = parse_annotation_file(json_path)
                # Debug: print first few failures (only for first 3 files)
                elif stats[split_name]['total'] <= 3:
                    print(f"⚠️ No JSON found for {img_file}")
                    print(f"   Tried: {json_candidates[0]}, {json_candidates[1]}")

            # Load image to get dimensions - validate image is readable
            if not validate_image(img_path):
                print(f"⚠️ Skipping invalid/corrupted image: {img_file}")
                continue

            try:
                img = Image.open(img_path)
                img_width, img_height = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Could not open image: {img_path} - {e}")
                continue

            # Create YOLO label file
            if len(annotations) > 0:
                stats[split_name]['with_annotations'] += 1

                # Determine output paths
                if split_name == 'train':
                    output_img_dir = train_images_dir
                    output_label_dir = train_labels_dir
                else:
                    output_img_dir = val_images_dir
                    output_label_dir = val_labels_dir

                # Copy image (already validated above)
                output_img_path = os.path.join(output_img_dir, img_file)
                shutil.copy2(img_path, output_img_path)

                # Create label file
                label_file = img_file.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
                label_path = os.path.join(output_label_dir, label_file)

                with open(label_path, 'w') as f:
                    for ann in annotations:
                        yolo_class = ann['yolo_class']
                        bbox = ann['bbox']
                        yolo_bbox = bbox_to_yolo(bbox, img_width, img_height)

                        # Only write valid bounding boxes
                        if yolo_bbox is not None:
                            # Write in YOLO format: class_id center_x center_y width height
                            f.write(f"{yolo_class} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")

                            stats[split_name]['class_counts'][yolo_class] += 1
            else:
                # Skip images without annotations
                pass

    # Print statistics
    print("\n" + "="*80)
    print("CONVERSION STATISTICS")
    print("="*80)
    for split_name in ['train', 'val']:
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats[split_name]['total']}")
        print(f"  Images with annotations: {stats[split_name]['with_annotations']}")
        print(f"  Class distribution:")
        for class_id in sorted(stats[split_name]['class_counts'].keys()):
            print(f"    {CLASS_NAMES[class_id]} (class {class_id}): {stats[split_name]['class_counts'][class_id]}")

    return stats

# Run conversion
if DATASET_BASE and os.path.exists(DATASET_BASE):
    print("Starting dataset conversion...")
    stats = convert_to_yolo_format(DATASET_BASE, OUTPUT_DIR, split='train', train_ratio=0.8)
    print("\n✅ Dataset conversion completed!")
else:
    print("⚠️ Dataset path not found. Please update DATASET_BASE variable.")


Starting dataset conversion...
Found 16988 image files
Found 8494 post-disaster images
Train: 6795, Val: 1699

Processing train split...


Converting train: 100%|██████████| 6795/6795 [09:48<00:00, 11.54it/s]



Processing val split...


Converting val: 100%|██████████| 1699/1699 [02:30<00:00, 11.30it/s]


CONVERSION STATISTICS

TRAIN:
  Total images: 6795
  Images with annotations: 3962
  Class distribution:
    No-Damage (class 0): 162251
    Moderate-Damage (class 1): 33263
    Total-Destruction (class 2): 13611

VAL:
  Total images: 1699
  Images with annotations: 984
  Class distribution:
    No-Damage (class 0): 41008
    Moderate-Damage (class 1): 9287
    Total-Destruction (class 2): 4588

✅ Dataset conversion completed!


In [30]:
# Create data.yaml file for YOLOv26
data_yaml_content = f"""# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: {os.path.abspath(OUTPUT_DIR)}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes
"""

# Save data.yaml
data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ Created data.yaml at: {data_yaml_path}")
print("\nConfiguration:")
print(data_yaml_content)


✅ Created data.yaml at: ./yolo_dataset/data.yaml

Configuration:
# YOLOv26 Dataset Configuration
# Building Damage Assessment Dataset

path: /home/jupyter/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

# Classes
names:
  0: No-Damage
  1: Moderate-Damage
  2: Total-Destruction

nc: 3  # number of classes



## Step 6: Train YOLOv26 Model

According to the research paper:
- **Epochs**: 50
- **Batch Size**: 800 (we'll use gradient accumulation for practical implementation)
- **Model**: YOLOv26 (we'll use YOLOv26n for faster training, or YOLOv26m for better accuracy)


In [ ]:
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)
# Training configuration
MODEL_SIZE = 'n'  # Options: 'n' (nano), 's' (small), 'm' (medium), 'l' (large), 'x' (xlarge)
EPOCHS = 10000
BATCH_SIZE = 16  # Adjust based on GPU memory (paper mentions 800, but that's likely with accumulation)
IMG_SIZE = 640  # YOLOv26 default input size

print("="*80)
print("🚀 TRAINING PREPARATION")
print("="*80)

# CRITICAL: Verify NumPy and OpenCV versions before training
import numpy as np
import cv2
numpy_version = np.__version__
opencv_version = cv2.__version__

print(f"\n📊 Environment Check (CRITICAL for debugging):")
print(f"   NumPy: {numpy_version}")
print(f"   OpenCV: {opencv_version}")

if numpy_version.startswith('2.'):
    print("\n❌ CRITICAL ERROR: NumPy 2.x detected!")
    print("   This WILL cause 'OpenCV imdecode' errors during training.")
    print("   Error message: 'buf is not a numpy array' in function 'imdecode'")
    print("   Root cause: OpenCV 4.8.x expects NumPy 1.x array format")
    print("   Solution: Restart kernel and run Cell 1 to fix NumPy version.")
    raise RuntimeError(f"NumPy 2.x ({numpy_version}) incompatible. Need 1.26.4")

if opencv_version.startswith('4.12') or opencv_version.startswith('4.9'):
    print(f"\n❌ CRITICAL ERROR: OpenCV {opencv_version} detected!")
    print("   This version requires NumPy 2.x, causing incompatibility.")
    print("   Solution: Restart kernel and run Cell 1 to fix OpenCV version.")
    raise RuntimeError(f"OpenCV {opencv_version} incompatible. Need 4.8.1.78")

if not (numpy_version.startswith('1.26') and opencv_version.startswith('4.8')):
    print(f"\n⚠️ WARNING: Version mismatch may cause issues!")
    print(f"   Expected: NumPy 1.26.x + OpenCV 4.8.x")
    print(f"   Got: NumPy {numpy_version} + OpenCV {opencv_version}")
else:
    print("   ✅ Versions compatible!")

# Check if dataset exists
if not os.path.exists(data_yaml_path):
    print(f"\n❌ ERROR: Dataset configuration not found: {data_yaml_path}")
    print("   Please run the dataset conversion cells first.")
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}")

print(f"\n✅ Dataset configuration found: {data_yaml_path}")

# Check dataset directories
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_label_dir = os.path.join(OUTPUT_DIR, "labels", "val")

print(f"\n📁 Dataset Structure Check:")
print(f"   Train images: {train_img_dir} - {'✅' if os.path.exists(train_img_dir) else '❌'}")
print(f"   Val images: {val_img_dir} - {'✅' if os.path.exists(val_img_dir) else '❌'}")
print(f"   Train labels: {train_label_dir} - {'✅' if os.path.exists(train_label_dir) else '❌'}")
print(f"   Val labels: {val_label_dir} - {'✅' if os.path.exists(val_label_dir) else '❌'}")

if os.path.exists(train_img_dir):
    train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Train images count: {len(train_imgs)}")
if os.path.exists(val_img_dir):
    val_imgs = [f for f in os.listdir(val_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"   Val images count: {len(val_imgs)}")

# Initialize YOLOv26 model
print(f"\n🤖 Initializing YOLOv26{MODEL_SIZE} model...")
try:
    model = YOLO(f'yolo26{MODEL_SIZE}.pt')  # Load pretrained weights
    print("   ✅ Model loaded successfully")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    raise

print(f"\n🚀 Starting training...")
print(f"   Model: YOLOv26{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}")

# Train the model with error handling
# Additional fix: ensure OpenCV compatibility by setting environment variable
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_DISABLE_OPENCL'] = '1'

print("\n" + "="*80)
print("⚠️  DEBUGGING INFO:")
print("   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible")
print(f"   Current: NumPy {numpy_version}, OpenCV {opencv_version}")
print("   Solution: Restart kernel + run Cell 1 to fix versions")
print("="*80 + "\n")

try:
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='building_damage_yolov26',
        project='./runs/detect',
        patience=10,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        val=True,
        plots=True,  # Generate training plots
        verbose=True,
        amp=False,  # Disable AMP to avoid check error
        workers=0,  # Set workers to 0 to avoid multiprocessing issues with image loading
        cache=False,  # Disable caching to avoid corrupted cache issues
        task='detect'  # Explicitly set task
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: ./runs/detect/building_damage_yolov26/")

except Exception as e:
    error_msg = str(e)
    print("\n" + "="*80)
    print("❌ TRAINING ERROR DETECTED")
    print("="*80)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {error_msg}")

    # Check for specific error patterns
    if 'imdecode' in error_msg.lower() or 'bad argument' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: OpenCV imdecode error detected!")
        print("   This is a NumPy/OpenCV version incompatibility issue.")
        print(f"   Current versions: NumPy {numpy_version}, OpenCV {opencv_version}")
        print("\n   Root cause:")
        print("   - OpenCV 4.12.0+ requires NumPy 2.x")
        print("   - OpenCV 4.8.x requires NumPy 1.x")
        print("   - Mismatch causes 'buf is not a numpy array' error")
        print("\n   Solution:")
        print("   1. Restart the kernel (Kernel -> Restart)")
        print("   2. Run Cell 1 again to fix package versions")
        print("   3. Verify NumPy 1.26.4 and OpenCV 4.8.1.78 are installed")
        print("   4. Run this cell again")
    elif 'numpy' in error_msg.lower() or 'ufunc' in error_msg.lower():
        print("\n🔍 DIAGNOSIS: NumPy compatibility error detected!")
        print(f"   Current NumPy version: {numpy_version}")
        print("   Solution: Restart kernel and run Cell 1 to fix NumPy version")
    else:
        print("\n🔍 DIAGNOSIS: Unknown error - check full traceback above")
        print("   This may be a dataset or configuration issue.")

    print("\n" + "="*80)
    print("Attempting retry with same settings (unlikely to help if version issue)...")
    print("="*80)

    # Retry with same settings (won't help if it's a version issue, but worth trying)
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name='building_damage_yolov262',
            project='./runs/detect',
            patience=10,
            save=True,
            save_period=10,
            val=True,
            plots=True,
            verbose=True,
            amp=False,
            workers=0,
            cache=False,
            task='detect'
        )

        print("\n✅ Training completed on retry!")
        print(f"📁 Results saved to: ./runs/detect/building_damage_yolov262/")

    except Exception as e2:
        print(f"\n❌ Retry also failed: {e2}")
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS:")
        print("="*80)
        print("Training failed after retry. Most likely causes:")
        print("1. NumPy/OpenCV version mismatch (most common)")
        print("2. Corrupted image files in dataset")
        print("3. Dataset path or structure issues")
        print("\nNext steps:")
        print("- Check the version diagnostics above")
        print("- Restart kernel and run Cell 1 to fix versions")
        print("- Verify dataset structure is correct")
        print("="*80)


🚀 TRAINING PREPARATION

📊 Environment Check (CRITICAL for debugging):
   NumPy: 1.26.4
   OpenCV: 4.8.1
   ✅ Versions compatible!

✅ Dataset configuration found: ./yolo_dataset/data.yaml

📁 Dataset Structure Check:
   Train images: ./yolo_dataset/images/train - ✅
   Val images: ./yolo_dataset/images/val - ✅
   Train labels: ./yolo_dataset/labels/train - ✅
   Val labels: ./yolo_dataset/labels/val - ✅
   Train images count: 4337
   Val images count: 1325

🤖 Initializing YOLOv26n model...
   ✅ Model loaded successfully

🚀 Starting training...
   Model: YOLOv26n
   Epochs: 10000
   Batch Size: 16
   Image Size: 640

⚠️  DEBUGGING INFO:
   If you see 'imdecode' error below, it means NumPy/OpenCV versions are incompatible
   Current: NumPy 1.26.4, OpenCV 4.8.1
   Solution: Restart kernel + run Cell 1 to fix versions

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22478MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaug

## Step 7: Model Evaluation

Evaluate the trained model using metrics from the research paper:
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")
# Load the best model
best_model_path = "./runs/detect/runs/detect/building_damage_yolov26/weights/best.pt"



if os.path.exists(best_model_path):
    print(f"✅ Loading best model: {best_model_path}")
    model = YOLO(best_model_path)

    # Validate the model
    print("\n📊 Running validation...")
    metrics = model.val(data=data_yaml_path)

    print("\n" + "="*80)
    print("VALIDATION METRICS")
    print("="*80)
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    # Print per-class metrics
    if hasattr(metrics.box, 'maps'):
        print("\nPer-class mAP50:")
        for i, class_name in CLASS_NAMES.items():
            if i < len(metrics.box.maps):
                print(f"  {class_name}: {metrics.box.maps[i]:.4f}")

    # Get detailed metrics
    print("\nDetailed Metrics:")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

else:
    print("⚠️ Best model not found. Please train the model first.")


## Step 8: Generate Predictions and Visualizations

Create visualizations similar to the research paper:
- Model predictions on test images
- Confusion matrix
- Precision-Recall curves
- F1-Confidence curves


In [ ]:
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")
# Generate predictions on validation set
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    # Get validation images
    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")

    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Found {len(val_images)} validation images")
        print("\nGenerating predictions on sample images...")

        # Predict on a few sample images
        sample_images = val_images[:5]  # First 5 images

        for img_file in sample_images:
            img_path = os.path.join(val_images_dir, img_file)

            # Run prediction
            results = model.predict(
                source=img_path,
                conf=0.25,  # Confidence threshold
                save=True,
                save_txt=True,
                save_conf=True,
                line_width=2
            )

            print(f"✅ Processed: {img_file}")

        print(f"\n📁 Predictions saved to: ./runs/detect/predict/")
    else:
        print("⚠️ Validation images directory not found.")
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")
# Create detailed evaluation plots
import matplotlib.pyplot as plt
# Try to import seaborn, but handle numpy/scipy compatibility issues
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except (ValueError, ImportError) as e:
    print(f"⚠️ Seaborn not available due to compatibility issues: {e}")
    print("   Continuing without seaborn - using matplotlib only")
    SEABORN_AVAILABLE = False

# Try to import sklearn metrics; skip evaluation plots if unavailable
try:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    print(f"⚠️ scikit-learn not available: {e}")
    print("   Skipping advanced evaluation plots that require sklearn")
    SKLEARN_AVAILABLE = False

import numpy as np

if SKLEARN_AVAILABLE:
    def plot_confusion_matrix_from_predictions(model, data_yaml_path, output_path=None):
        """Generate confusion matrix from model predictions"""
        try:
            # Run validation to get predictions
            results = model.val(data=data_yaml_path, plots=True, save_json=True)

            # The validation already generates confusion matrix
            # Check if confusion matrix image exists
            confusion_path = "./runs/detect/building_damage_yolov26/confusion_matrix.png"
            if os.path.exists(confusion_path):
                print(f"✅ Confusion matrix saved at: {confusion_path}")

                # Display it
                img = plt.imread(confusion_path)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title('Confusion Matrix', fontsize=16, pad=20)
                plt.tight_layout()
                if output_path:
                    plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("⚠️ Confusion matrix not found. Running validation with plots...")
                model.val(data=data_yaml_path, plots=True)

        except Exception as e:
            print(f"Error generating confusion matrix: {e}")
else:
    def plot_confusion_matrix_from_predictions(*args, **kwargs):
        print("⚠️ Skipping confusion matrix plotting because scikit-learn is unavailable.")

# Generate confusion matrix
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    plot_confusion_matrix_from_predictions(model, data_yaml_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")
# Create damage distribution visualization
def plot_damage_distribution(predictions_dir):
    """Plot distribution of predicted damage classes"""
    try:
        # Collect predictions from saved text files
        class_counts = Counter()

        pred_labels_dir = os.path.join(predictions_dir, "labels")
        if os.path.exists(pred_labels_dir):
            for label_file in os.listdir(pred_labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(pred_labels_dir, label_file)
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1

        if len(class_counts) > 0:
            # Create bar chart
            classes = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
            counts = [class_counts[i] for i in sorted(class_counts.keys())]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(classes, counts, color=['green', 'orange', 'red'])
            plt.xlabel('Damage Category', fontsize=12)
            plt.ylabel('Number of Instances', fontsize=12)
            plt.title('Distribution of Predicted Damage Categories', fontsize=14, fontweight='bold')
            plt.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar, count in zip(bars, counts):
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(count)}',
                        ha='center', va='bottom', fontsize=11)

            plt.tight_layout()
            plt.savefig('./damage_distribution.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\nDamage Distribution:")
            for class_id in sorted(class_counts.keys()):
                print(f"  {CLASS_NAMES[class_id]}: {class_counts[class_id]}")
        else:
            print("⚠️ No predictions found to plot.")

    except Exception as e:
        print(f"Error plotting damage distribution: {e}")

# Plot damage distribution
predictions_dir = "./runs/detect/predict"
if os.path.exists(predictions_dir):
    plot_damage_distribution(predictions_dir)
else:
    print("⚠️ Predictions directory not found. Run predictions first.")


## Step 9: Generate Heatmaps (Post-Disaster Analysis)

As mentioned in the research paper, heatmaps are created to visualize damage levels across the affected area.


In [ ]:
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")
def create_damage_heatmap(model, image_path, output_path=None, conf_threshold=0.25):
    """
    Create a heatmap showing damage levels across the image

    Args:
        model: YOLOv26 model
        image_path: Path to input image
        output_path: Path to save heatmap
        conf_threshold: Confidence threshold for predictions
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Create heatmap (initialize with zeros)
        heatmap = np.zeros((h, w), dtype=np.float32)

        # Run prediction
        results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

        # Process predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Get class and confidence
                class_id = int(box.cls[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())

                # Damage intensity: 0=No-Damage, 1=Moderate, 2=Total-Destruction
                # Map to heatmap intensity (0-1 scale)
                if class_id == 0:  # No-Damage
                    intensity = 0.2
                elif class_id == 1:  # Moderate-Damage
                    intensity = 0.6
                else:  # Total-Destruction
                    intensity = 1.0

                # Weight by confidence
                intensity *= conf

                # Add to heatmap (use bounding box area)
                heatmap[y1:y2, x1:x2] = np.maximum(heatmap[y1:y2, x1:x2], intensity)

        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # Original image
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(img_rgb, alpha=0.7)
        im = axes[1].imshow(heatmap, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=1)
        axes[1].set_title('Damage Heatmap\n(Red: High Damage, Green: Low/No Damage)',
                         fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Damage Intensity', rotation=270, labelpad=20)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Heatmap saved to: {output_path}")

        plt.show()

    except Exception as e:
        print(f"Error creating heatmap: {e}")

# Generate heatmaps for sample images
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_images_dir = os.path.join(OUTPUT_DIR, "images", "val")
    if os.path.exists(val_images_dir):
        val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        # Create heatmaps for first 3 images
        for i, img_file in enumerate(val_images[:3]):
            img_path = os.path.join(val_images_dir, img_file)
            output_path = f'./heatmap_{i+1}_{os.path.splitext(img_file)[0]}.png'
            print(f"\nGenerating heatmap for: {img_file}")
            create_damage_heatmap(model, img_path, output_path)
else:
    print("⚠️ Model not found. Please train the model first.")


In [ ]:
# Print summary
print("="*80)
print("IMPLEMENTATION SUMMARY")
print("="*80)

print("\n✅ Completed Steps:")
print("  1. Dataset exploration and understanding")
print("  2. Annotation parsing (JSON/GeoJSON)")
print("  3. Polygon to bounding box conversion")
print("  4. Damage class mapping (5 classes → 3 classes)")
print("  5. YOLO format dataset preparation")
print("  6. Train/validation split (80/20)")
print("  7. YOLOv26 model training")
print("  8. Model evaluation and metrics")
print("  9. Prediction visualization")
print("  10. Damage heatmap generation")

print("\n📊 Model Configuration:")
print(f"  Architecture: YOLOv26{MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")

print("\n📁 Output Files:")
print(f"  Dataset: {OUTPUT_DIR}")
print(f"  Model: ./runs/detect/building_damage_yolov26/weights/best.pt")
print(f"  Results: ./runs/detect/building_damage_yolov26/")

print("\n📈 Next Steps:")
print("  - Fine-tune hyperparameters for better performance")
print("  - Experiment with different YOLOv26 model sizes")
print("  - Add data augmentation techniques")
print("  - Evaluate on test set")
print("  - Compare results with research paper metrics")

print("\n" + "="*80)
